# HK01 news scraper (Traditional Chinese)

This notebook downloads HK01 news articles from https://www.hk01.com/ for an inclusive date range.

It creates exactly two outputs:

- News text files (content only): `C:\Program Files\Studying\coding\RAG_project\data\hk01_news`
- One metadata CSV for the run: `C:\Program Files\Studying\coding\RAG_project\data\hk01_news_metadata`

Main function:

`scrape_hk01_news(start_date, end_date)`

Date format: `DD-MM-YYYY` (for example, `"01-05-2025"`).
For one day, use the same date for both arguments.

In [1]:
from __future__ import annotations

import csv
import html
import json
import re
from datetime import date, datetime
from html.parser import HTMLParser
from pathlib import Path
from time import sleep
from typing import Any
from urllib.parse import urlparse
from zoneinfo import ZoneInfo

import requests

PROJECT_ROOT = Path(r"C:\Program Files\Studying\coding\RAG_project")
NEWS_DIR = PROJECT_ROOT / "data" / "hk01_news"
METADATA_DIR = PROJECT_ROOT / "data" / "hk01_news_metadata"

HK01_SITEMAP_URL = "https://www.hk01.com/sitemapByLastMod.xml"
HK_TIMEZONE = ZoneInfo("Asia/Hong_Kong")


class HtmlToTextParser(HTMLParser):
    """Convert HTML fragments into readable plain text."""

    def __init__(self) -> None:
        super().__init__(convert_charrefs=True)
        self._parts: list[str] = []
        self._skip_depth = 0
        self._block_tags = {"p", "div", "br", "li", "h1", "h2", "h3", "h4", "blockquote"}
        self._skip_tags = {"script", "style", "noscript", "iframe", "svg", "figure", "figcaption"}

    def handle_starttag(self, tag: str, attrs: list[tuple[str, str | None]]) -> None:
        if self._skip_depth > 0:
            self._skip_depth += 1
            return

        if tag in self._skip_tags:
            self._skip_depth = 1
            return

        if tag in self._block_tags:
            self._parts.append("\n")

    def handle_endtag(self, tag: str) -> None:
        if self._skip_depth > 0:
            self._skip_depth -= 1
            return

        if tag in self._block_tags:
            self._parts.append("\n")

    def handle_data(self, data: str) -> None:
        if self._skip_depth == 0:
            self._parts.append(data)

    def text(self) -> str:
        raw_text = html.unescape("".join(self._parts))
        lines = [re.sub(r"\s+", " ", line).strip() for line in raw_text.splitlines()]
        return "\n\n".join(line for line in lines if line)


def parse_scrape_date(value: date | str) -> date:
    """Parse input date. Required format is DD-MM-YYYY."""
    if isinstance(value, date):
        return value

    for date_format in ("%d-%m-%Y", "%Y-%m-%d"):
        try:
            return datetime.strptime(value, date_format).date()
        except ValueError:
            pass

    raise ValueError("Use date format DD-MM-YYYY, for example '01-05-2025'.")


def safe_article_filename(value: str, max_length: int = 120) -> str:
    """Create a Windows-safe filename from article title."""
    value = html.unescape(value).strip()
    value = re.sub(r'[<>:"/\\|?*]', "", value)
    value = re.sub(r"\s+", " ", value).strip(" .")
    return value[:max_length].strip(" .") or "Untitled article"


def strip_url_query_and_fragment(url: str) -> str:
    """Normalize article URL by removing query parameters and fragment."""
    parsed = urlparse(url)
    return f"{parsed.scheme}://{parsed.netloc}{parsed.path}"


def parse_hk01_iso_datetime(value: str) -> datetime:
    """Parse ISO datetime string and convert to Hong Kong timezone."""
    parsed = datetime.fromisoformat(value.replace("Z", "+00:00"))
    if parsed.tzinfo is None:
        parsed = parsed.replace(tzinfo=HK_TIMEZONE)
    return parsed.astimezone(HK_TIMEZONE)


def extract_sitemap_candidates(timeout_seconds: int = 30) -> list[dict[str, str]]:
    """Read hk01 sitemap and return candidate articles with URL, date, and title."""
    response = requests.get(HK01_SITEMAP_URL, timeout=timeout_seconds)
    response.raise_for_status()
    xml_text = response.text

    # Parse each <url> node from sitemap while keeping date and title for metadata fallback.
    url_blocks = re.findall(r"<url>(.*?)</url>", xml_text, flags=re.DOTALL)
    candidates: list[dict[str, str]] = []

    for block in url_blocks:
        loc_match = re.search(r"<loc>(.*?)</loc>", block)
        date_match = re.search(r"<news:publication_date>(.*?)</news:publication_date>", block)
        title_match = re.search(r"<news:title>(.*?)</news:title>", block)

        if not loc_match or not date_match:
            continue

        raw_url = html.unescape(loc_match.group(1).strip())
        if "/article/" not in raw_url:
            continue

        candidates.append(
            {
                "url": strip_url_query_and_fragment(raw_url),
                "sitemap_publication_date": date_match.group(1).strip(),
                "sitemap_title": html.unescape(title_match.group(1).strip()) if title_match else "",
            }
        )

    # Deduplicate because sitemap can contain repeated URLs.
    dedup: dict[str, dict[str, str]] = {}
    for item in candidates:
        dedup[item["url"]] = item

    return list(dedup.values())


def extract_next_data_json(html_text: str) -> dict[str, Any]:
    """Extract __NEXT_DATA__ JSON object from an hk01 article page."""
    match = re.search(
        r'<script id="__NEXT_DATA__" type="application/json">(.*?)</script>',
        html_text,
        flags=re.DOTALL,
    )
    if not match:
        raise ValueError("Cannot find __NEXT_DATA__ JSON in article page.")

    return json.loads(match.group(1))


def find_primary_article_object(node: Any) -> dict[str, Any] | None:
    """Find the main article object containing blocks, categories, title, and publish time."""
    if isinstance(node, dict):
        has_article_shape = (
            isinstance(node.get("blocks"), list)
            and isinstance(node.get("categories"), list)
            and node.get("title")
            and node.get("publishTime")
        )
        if has_article_shape:
            return node

        for value in node.values():
            found = find_primary_article_object(value)
            if found:
                return found

    elif isinstance(node, list):
        for item in node:
            found = find_primary_article_object(item)
            if found:
                return found

    return None


def flatten_html_tokens(tokens: Any) -> str:
    """Convert hk01 htmlTokens structure into plain text paragraphs."""
    paragraphs: list[str] = []

    # htmlTokens is usually a nested list. Traverse recursively and keep token content in order.
    def walk(node: Any, current_fragments: list[str]) -> None:
        if isinstance(node, dict):
            content = node.get("content")
            if isinstance(content, str) and content.strip():
                current_fragments.append(content)
            return

        if isinstance(node, list):
            local_fragments: list[str] = []
            for item in node:
                walk(item, local_fragments)

            joined = "".join(local_fragments).strip()
            if joined:
                paragraphs.append(joined)

    walk(tokens, [])

    cleaned: list[str] = []
    for paragraph in paragraphs:
        normalized = re.sub(r"\s+", " ", paragraph).strip()
        if normalized:
            cleaned.append(normalized)

    return "\n\n".join(cleaned)


def extract_article_text_from_blocks(article_obj: dict[str, Any]) -> str:
    """Build article content text from hk01 blocks, keeping only text-like blocks."""
    text_parts: list[str] = []

    teaser = article_obj.get("teaser")
    if isinstance(teaser, list):
        teaser_text = "\n\n".join(re.sub(r"\s+", " ", str(item)).strip() for item in teaser if str(item).strip())
        if teaser_text:
            text_parts.append(teaser_text)

    for block in article_obj.get("blocks", []):
        block_type = str(block.get("blockType", "")).lower()

        # Keep only textual block types to avoid image/code/related clutter.
        if block_type == "text":
            block_text = flatten_html_tokens(block.get("htmlTokens", []))
            if block_text:
                text_parts.append(block_text)
            continue

        if block_type == "code":
            html_string = block.get("htmlString", "")
            if isinstance(html_string, str) and html_string.strip():
                parser = HtmlToTextParser()
                parser.feed(html_string)
                parser.close()
                code_text = parser.text()
                if code_text and len(code_text) > 40 and "http" not in code_text.lower():
                    text_parts.append(code_text)

    raw_text = "\n\n".join(part for part in text_parts if part.strip())

    # Final cleanup: remove obvious social/share/related clutter lines.
    cleaned_lines: list[str] = []
    for line in raw_text.splitlines():
        normalized = re.sub(r"\s+", " ", line).strip()
        if not normalized:
            continue

        lowered = normalized.lower()
        if lowered in {"分享", "facebook", "whatsapp", "telegram", "x", "twitter", "wechat", "line"}:
            continue
        if lowered.startswith(("相關文章", "延伸閱讀", "更多", "立即訂閱")):
            continue

        cleaned_lines.append(normalized)

    return "\n\n".join(cleaned_lines)


def scrape_hk01_news(start_date: date | str, end_date: date | str) -> tuple[list[Path], Path]:
    """
    Scrape hk01 news articles for an inclusive date range.

    Parameters
    ----------
    start_date:
        First date to scrape. Use DD-MM-YYYY, for example '01-01-2025'.
    end_date:
        Last date to scrape. Use DD-MM-YYYY, for example '01-05-2025'.
        For one day, pass the same value for start_date and end_date.

    Returns
    -------
    tuple[list[Path], Path]
        First output: news text file paths saved in NEWS_DIR.
        Second output: one metadata CSV file path saved in METADATA_DIR.
    """
    # Validate date range first.
    start = parse_scrape_date(start_date)
    end = parse_scrape_date(end_date)
    if start > end:
        raise ValueError("start_date must be earlier than or equal to end_date.")

    # Ensure output folders exist.
    NEWS_DIR.mkdir(parents=True, exist_ok=True)
    METADATA_DIR.mkdir(parents=True, exist_ok=True)

    session = requests.Session()
    session.headers.update(
        {
            "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) RAG-project-hk01-scraper/1.0",
            "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
        }
    )

    candidates = extract_sitemap_candidates()
    saved_news_files: list[Path] = []
    metadata_rows: list[dict[str, str]] = []
    filename_counts: dict[str, int] = {}

    for item in candidates:
        sitemap_dt = parse_hk01_iso_datetime(item["sitemap_publication_date"])
        if not (start <= sitemap_dt.date() <= end):
            continue

        article_url = item["url"]

        try:
            response = session.get(article_url, timeout=30)
            response.raise_for_status()
            next_data = extract_next_data_json(response.text)
            article_obj = find_primary_article_object(next_data)
            if not article_obj:
                print(f"Skipped (article object not found): {article_url}")
                continue

            # Prefer article publish time from page payload, fallback to sitemap date.
            publish_ts = article_obj.get("publishTime")
            if isinstance(publish_ts, (int, float, str)) and str(publish_ts).isdigit():
                published_hk = datetime.fromtimestamp(int(publish_ts), tz=HK_TIMEZONE)
            else:
                published_hk = sitemap_dt

            if not (start <= published_hk.date() <= end):
                continue

            title = html.unescape(str(article_obj.get("title") or item.get("sitemap_title") or "Untitled article"))
            article_text = extract_article_text_from_blocks(article_obj)

            if not article_text.strip():
                print(f"Skipped empty content: {article_url}")
                continue

            categories_list = []
            for category in article_obj.get("categories", []):
                name = category.get("publishName") or category.get("name")
                if isinstance(name, str) and name.strip():
                    categories_list.append(html.unescape(name.strip()))

            date_prefix = published_hk.strftime("%d-%m-%Y")
            filename_stem = f"{date_prefix}_{safe_article_filename(title)}"

            # Keep filename unique when titles repeat on the same day.
            filename_counts[filename_stem] = filename_counts.get(filename_stem, 0) + 1
            if filename_counts[filename_stem] > 1:
                filename_stem = f"{filename_stem}_{filename_counts[filename_stem]}"

            news_file = NEWS_DIR / f"{filename_stem}.txt"
            news_file.write_text(article_text.strip() + "\n", encoding="utf-8")
            saved_news_files.append(news_file)

            metadata_rows.append(
                {
                    "articles": title,
                    "released_date": published_hk.strftime("%d-%m-%Y"),
                    "categories": "; ".join(dict.fromkeys(categories_list)),
                    "url": article_url,
                }
            )

            # Be polite to the server.
            sleep(0.2)

        except Exception as exc:
            print(f"Skipped {article_url} due to error: {exc}")
            continue

    metadata_csv = METADATA_DIR / f"hk01_metadata_{start.strftime('%d-%m-%Y')}_to_{end.strftime('%d-%m-%Y')}.csv"
    with metadata_csv.open("w", newline="", encoding="utf-8-sig") as csv_file:
        fieldnames = ["articles", "released_date", "categories", "url"]
        writer = csv.DictWriter(csv_file, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(metadata_rows)

    print(f"Saved {len(saved_news_files)} article text files to: {NEWS_DIR}")
    print(f"Saved metadata CSV to: {metadata_csv}")
    return saved_news_files, metadata_csv

In [2]:
# Hotfix: hk01 sitemap URLs are category-style paths (for example /即時體育/60361209/...),
# not /article/... paths. This override keeps all article-like URLs with a numeric ID segment.
def extract_sitemap_candidates(timeout_seconds: int = 30) -> list[dict[str, str]]:
    """Read hk01 sitemap and return candidate articles with URL, date, and title."""
    response = requests.get(HK01_SITEMAP_URL, timeout=timeout_seconds)
    response.raise_for_status()
    xml_text = response.text

    url_blocks = re.findall(r"<url>(.*?)</url>", xml_text, flags=re.DOTALL)
    candidates: list[dict[str, str]] = []

    for block in url_blocks:
        loc_match = re.search(r"<loc>(.*?)</loc>", block)
        date_match = re.search(r"<news:publication_date>(.*?)</news:publication_date>", block)
        title_match = re.search(r"<news:title>(.*?)</news:title>", block)

        if not loc_match or not date_match:
            continue

        raw_url = html.unescape(loc_match.group(1).strip())
        normalized_url = strip_url_query_and_fragment(raw_url)

        # Keep hk01 content URLs with a numeric article ID in path.
        if not re.search(r"/\d{5,}(/|$)", normalized_url):
            continue

        candidates.append(
            {
                "url": normalized_url,
                "sitemap_publication_date": date_match.group(1).strip(),
                "sitemap_title": html.unescape(title_match.group(1).strip()) if title_match else "",
            }
        )

    dedup: dict[str, dict[str, str]] = {}
    for item in candidates:
        dedup[item["url"]] = item
        
    return list(dedup.values())

## Run scraper

Use DD-MM-YYYY dates. The range is inclusive.

If you only want one day, set start_date and end_date to the same value.

Note:
This scraper currently relies on HK01 sitemap feeds. If HK01 does not expose your requested historical window in sitemap output, the result can be 0 files.

In [3]:
# Example
# Pick a recent range likely to exist in HK01 sitemap output.
news_files, metadata_csv = scrape_hk01_news("30-06-2026", "30-06-2026")

print(f"News text files: {len(news_files)}")
print(f"Metadata CSV: {metadata_csv}")

Saved 375 article text files to: C:\Program Files\Studying\coding\RAG_project\data\hk01_news
Saved metadata CSV to: C:\Program Files\Studying\coding\RAG_project\data\hk01_news_metadata\hk01_metadata_30-06-2026_to_30-06-2026.csv
News text files: 375
Metadata CSV: C:\Program Files\Studying\coding\RAG_project\data\hk01_news_metadata\hk01_metadata_30-06-2026_to_30-06-2026.csv


In [15]:
candidates = extract_sitemap_candidates()
candidate_dates = sorted(parse_hk01_iso_datetime(item["sitemap_publication_date"]).date() for item in candidates)

print(f"Sitemap candidates: {len(candidates)}")
if candidate_dates:
    print(f"Oldest date in source: {candidate_dates[0]}")
    print(f"Newest date in source: {candidate_dates[-1]}")

start = parse_scrape_date("01-01-2026")
end = parse_scrape_date("10-01-2026")
count_in_range = sum(1 for d in candidate_dates if start <= d <= end)
print(f"Candidates between 01-01-2026 and 10-01-2026: {count_in_range}")

Sitemap candidates: 1000
Oldest date in source: 2018-03-27
Newest date in source: 2026-06-25
Candidates between 01-01-2026 and 10-01-2026: 0
